# FTIR Analysis of Honey Samples

This notebook reads ATR-FTIR spectra from the `data/honey/` folder, applies blank correction,
plots the spectra (overlay and waterfall), and runs PCA-based chemometric analysis.

---

## Background: ATR-FTIR spectroscopy of honey

Attenuated Total Reflectance Fourier-Transform Infrared (ATR-FTIR) spectroscopy measures how
much infrared light a sample absorbs at each wavenumber (cm⁻¹). Different chemical bonds
absorb at characteristic wavenumbers, so a spectrum is a chemical 'fingerprint' of the sample.

For honey authentication, the most informative region is the **carbohydrate fingerprint
(1800–650 cm⁻¹)**:
- ~1640 cm⁻¹ : O-H bending of water
- ~1415 cm⁻¹ : C-H / O-H deformation
- ~1155 cm⁻¹ : C-O-C stretch (sugar ring, diagnostic for disaccharides such as sucrose)
- ~1080 cm⁻¹ : C-OH stretch (glucose)
- ~1025 cm⁻¹ : C-OH stretch (fructose)
- ~930  cm⁻¹ : pyranose / furanose ring vibrations

---

## Data format

CSV files exported from the Perkin-Elmer Spectrum instrument:
```
Line 1 : Created as New Dataset, Sample NNN By iruser Date ...
Line 2 : cm-1,%T
Line 3+: 4200.00,99.90   ← wavenumber (cm⁻¹), % transmittance
```

Wavenumbers run from **4200 → 650 cm⁻¹** (3551 points, 1 cm⁻¹ spacing).
Files whose names start with `blank` are background spectra; all others are honey samples.

---

## Running on Google Colab

1. Open this notebook in Colab.
2. Use the file browser (📁 left panel) to create `data/honey/` under `/content/`.
3. Upload your `.csv` files into that folder.
4. Run all cells (`Runtime → Run all`).

Alternatively, mount Google Drive — see the commented code in Section 0.

## 0 · Environment setup

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 · Environment setup
#
# Detects whether we are on Google Colab or running locally, then sets:
#   DATA_DIR   – where the instrument CSV files live
#   OUTPUT_DIR – where figures are saved  (data/processed/)
# ─────────────────────────────────────────────────────────────────────────────

import sys
from pathlib import Path

# Try to import the google.colab module.  It is only present when the kernel
# is actually running inside Colab; the import silently fails on a local machine.
try:
    import google.colab      # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    # ── Google Colab paths ────────────────────────────────────────────────────
    # Files uploaded via the Colab file browser land under /content/.
    # We expect:
    #   /content/data/honey/      ← upload .csv files here
    #   /content/data/processed/  ← created automatically for output figures
    #
    # To use Google Drive instead, uncomment the four lines below.
    # Drive is mounted at /content/drive/MyDrive/.
    # -------------------------------------------------------------------------
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DATA_DIR   = Path('/content/drive/MyDrive/YOUR_FOLDER/data/honey')
    # OUTPUT_DIR = Path('/content/drive/MyDrive/YOUR_FOLDER/data/processed')
    # -------------------------------------------------------------------------
    DATA_DIR   = Path('/content/data/honey')
    OUTPUT_DIR = Path('/content/data/processed')
    print("▶ Running on Google Colab")
else:
    # ── Local paths ───────────────────────────────────────────────────────────
    # This notebook lives in  <project>/notebooks/
    # Data lives in           <project>/data/honey/
    # Output goes to          <project>/data/processed/
    # Path('..') navigates from notebooks/ up one level to the project root.
    DATA_DIR   = Path('../data/honey')
    OUTPUT_DIR = Path('../data/processed')
    print("▶ Running locally")

# Show resolved absolute paths so the user can verify them immediately
print(f"   Data directory   : {DATA_DIR.resolve()}")
print(f"   Output directory : {OUTPUT_DIR.resolve()}")

# Create the output directory (mkdir is a no-op if it already exists,
# and parents=True creates any missing intermediate directories)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("\nOutput directory ready ✓")

## 1 · Imports

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 · Imports
#
# All packages below are pre-installed in Google Colab and in standard
# scientific Python environments (Anaconda, pip install numpy pandas
# matplotlib scikit-learn).
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np                       # Numerical arrays and vectorised maths
import pandas as pd                      # Reading and manipulating CSV data
import matplotlib.pyplot as plt          # All plotting
import matplotlib.cm as cm               # Colour maps for multi-line spectral plots
from sklearn.decomposition import PCA    # Principal Component Analysis

# Render plots inline (embedded in notebook output cell).
# This works in both Colab and classic Jupyter / JupyterLab.
%matplotlib inline

# ── Global plot style ─────────────────────────────────────────────────────────
# Setting rcParams here once keeps every subsequent figure visually consistent
# without repeating style arguments in each individual plot command.
plt.rcParams.update({
    'figure.dpi'     : 120,   # Screen display resolution
    'savefig.dpi'    : 150,   # Resolution of saved PNG files
    'font.size'      : 11,
    'axes.titlesize' : 12,
    'axes.labelsize' : 11,
    'legend.fontsize': 9,
})

print("Imports OK ✓")

## 2 · Load spectra

We read every `.csv` file in `DATA_DIR`.
Files starting with `blank` → `blank_spectra` dict; everything else → `sample_spectra` dict.

### Why convert %T to Absorbance immediately?

The instrument records **% Transmittance (%T)** — the fraction of light that passes through.
We convert to **Absorbance (A)** before any further processing because:

- The **Beer-Lambert law** (A = ε·c·l) is *linear* with concentration.
  %T has an exponential relationship with concentration and is therefore unsuitable
  for multivariate statistics (PCA, regression), which assume linear relationships.
- Blank subtraction is only physically meaningful in absorbance units.

**Conversion:** A = −log₁₀(T) = −log₁₀(%T / 100) = **2 − log₁₀(%T)**

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 · Load spectra
# ─────────────────────────────────────────────────────────────────────────────

# ── Helper: read one instrument CSV ──────────────────────────────────────────

def read_ftir_csv(path: Path) -> tuple:
    """
    Read a Perkin-Elmer Spectrum CSV export and return numeric arrays.

    File layout
    -----------
    Line 1 : 'Created as New Dataset,Sample NNN By iruser Date ...'
    Line 2 : 'cm-1,%T'   ← column header, not data — must be skipped
    Line 3+: wavenumber,pct_T  (e.g. 4200.00,99.90)

    Returns
    -------
    wavenumbers : np.ndarray, shape (N,)  — cm⁻¹ axis, descending 4200→650
    pct_T       : np.ndarray, shape (N,)  — raw % transmittance
    meta        : dict  — 'filename', 'sample_number' (int|None), 'raw'
    """
    # Open with 'utf-8-sig' so that a BOM (byte-order mark) that Windows
    # software sometimes prepends is stripped automatically.
    with open(path, encoding='utf-8-sig') as f:
        meta_line = f.readline().strip()   # Line 1: instrument metadata string
        f.readline()                        # Line 2: 'cm-1,%T' header — discard

    # ── Parse instrument metadata ─────────────────────────────────────────────
    # The metadata string looks like:
    #   "Created as New Dataset,Sample 302 By iruser Date Friday, March 13 2026"
    # Split on ',' → parts[1] = "Sample 302 By iruser Date Friday"
    meta = {'raw': meta_line, 'filename': path.stem}
    try:
        sample_info = meta_line.split(',')[1].strip()   # "Sample 302 By ..."
        meta['sample_number'] = int(sample_info.split()[1])   # "302" → 302
    except (IndexError, ValueError):
        # Parsing failed (different export format, extra commas, etc.)
        # Continue without a sample number; blank matching will use the mean.
        meta['sample_number'] = None

    # ── Read numeric data ─────────────────────────────────────────────────────
    # skiprows=2 jumps over both header lines so line 3 is the first data row.
    # names= prevents pandas from treating line 3 as a header.
    # errors='coerce' converts stray non-numeric values to NaN (then dropped),
    # making the parser robust to minor formatting differences between files.
    df = pd.read_csv(path, skiprows=2, names=['wavenumber', 'pct_T'],
                     encoding='utf-8-sig')
    df['wavenumber'] = pd.to_numeric(df['wavenumber'], errors='coerce')
    df['pct_T']      = pd.to_numeric(df['pct_T'],      errors='coerce')
    df = df.dropna()   # remove rows that failed to parse

    return df['wavenumber'].values, df['pct_T'].values, meta


# ── Load all CSV files from DATA_DIR ──────────────────────────────────────────

# sorted() gives alphabetical order, which is consistent across operating systems
# (unlike os.listdir, which can vary by platform)
csv_files = sorted(DATA_DIR.glob('*.csv'))

if not csv_files:
    raise FileNotFoundError(
        f"No .csv files found in {DATA_DIR.resolve()}\n"
        "Check the DATA_DIR path in Section 0 and make sure files are uploaded."
    )

# Two dicts, keyed by filename stem (e.g. 'blank_1_130326', 'golden_130321').
# Each value is a 3-tuple: (wavenumber_array, absorbance_array, metadata_dict).
blank_spectra  = {}   # background / blank measurements
sample_spectra = {}   # honey (or other food) samples

for f in csv_files:
    wn, pct_T, meta = read_ftir_csv(f)

    # ── Convert %T → Absorbance ───────────────────────────────────────────────
    # np.clip sets a floor of 0.01 %T before taking the log.
    # Without clipping, instrument noise can drive %T to 0 or below, making
    # log10 undefined (−inf).  A floor of 0.01 %T → A ≈ 4.0, which is above
    # any real signal in these spectra, so no genuine data is altered.
    pct_T_safe = np.clip(pct_T, 0.01, None)
    absorbance = 2.0 - np.log10(pct_T_safe)   # A = 2 − log10(%T)

    entry = (wn, absorbance, meta)

    # Classify: blank files start with 'blank' (case-insensitive)
    if f.stem.lower().startswith('blank'):
        blank_spectra[f.stem] = entry
    else:
        sample_spectra[f.stem] = entry

# ── Summary ───────────────────────────────────────────────────────────────────
print(f"Blanks  ({len(blank_spectra)}):")
for name, (_, _, m) in blank_spectra.items():
    print(f"  sample #{str(m['sample_number']):>4}  →  {name}")

print(f"\nSamples ({len(sample_spectra)}):")
for name, (_, _, m) in sample_spectra.items():
    print(f"  sample #{str(m['sample_number']):>4}  →  {name}")

# Report the wavenumber axis (should be identical across all files)
wn_example = list(sample_spectra.values())[0][0]
print(f"\nWavenumber axis: {wn_example[-1]:.0f} – {wn_example[0]:.0f} cm⁻¹  "
      f"({len(wn_example)} points, "
      f"{abs(float(wn_example[0]) - float(wn_example[1])):.1f} cm⁻¹ spacing)")

## 3 · Blank correction

**Why do we need blank correction?**
Even a spotlessly clean ATR crystal picks up small contributions from atmospheric CO₂, water vapour, and instrument thermal emission. The blank spectrum (measured on the bare crystal just before or after each sample) captures this background. Subtracting it from the sample spectrum leaves only the honey signal.

**How to match blanks to samples?**
The sample numbers in the file metadata reveal that blanks and samples were recorded in alternating sequence (e.g. blank #299, Mix #300, blank #301, golden #302 …). The best correction therefore uses the blank with the **closest sample number** — a proxy
for the closest measurement time. If sample numbers cannot be parsed, the cell falls back to subtracting the mean of all blanks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 · Blank correction
# ─────────────────────────────────────────────────────────────────────────────

# ── Build look-up: sample_number → blank absorbance array ────────────────────
# Include only blanks whose sample_number was successfully parsed.
# Dict comprehension iterates over blank_spectra, unpacks each tuple,
# and filters on whether meta['sample_number'] is not None.
blank_by_num = {
    meta['sample_number']: abs_arr
    for _, (_, abs_arr, meta) in blank_spectra.items()
    if meta['sample_number'] is not None
}

# Pre-compute the element-wise mean across all blank spectra as a fallback
# (used when no sample number is available to identify the nearest blank).
mean_blank = np.mean(list(blank_by_num.values()), axis=0)


def nearest_blank_spectrum(sample_num) -> np.ndarray:
    """
    Return the blank absorbance array closest in sample number to sample_num.
    Falls back to the mean blank if blank_by_num is empty or sample_num is None.
    """
    if not blank_by_num or sample_num is None:
        return mean_blank
    # min() with a key function: find the blank number minimising |b - sample_num|
    closest_num = min(blank_by_num.keys(), key=lambda b: abs(b - sample_num))
    return blank_by_num[closest_num]


# ── Apply blank correction ─────────────────────────────────────────────────────
# 'corrected' is the main data store used throughout the rest of the notebook.
# Structure: { label_string : (wavenumber_array, corrected_absorbance_array) }
corrected = {}

print("Blank correction pairing:")
print(f"  {'Sample':<30}  {'sample#':>8}  →  {'blank#':>6}")
print("  " + "─" * 52)

for label, (wn, abs_arr, meta) in sample_spectra.items():
    s_num = meta['sample_number']

    # Select the blank absorbance array to subtract
    blank_arr = nearest_blank_spectrum(s_num)

    # Point-by-point subtraction: corrected[i] = sample[i] − blank[i]
    # This is valid because all spectra share the same wavenumber axis.
    corr = abs_arr - blank_arr
    corrected[label] = (wn, corr)

    # Identify the blank that was actually chosen (for the audit log)
    b_num = (min(blank_by_num.keys(), key=lambda b: abs(b - s_num))
             if blank_by_num and s_num is not None else 'mean')
    print(f"  {label:<30}  {str(s_num):>8}  →  {str(b_num):>6}")

print("\nBlank-corrected absorbance = sample absorbance − nearest blank ✓")

## 4 · Plot spectra

### Overlay vs waterfall — which to use for FTIR?

| Plot type | When to use | Advantage | Limitation |
|-----------|------------|-----------|------------|
| **Overlay** | Comparing peak positions and relative intensities | Peak heights directly comparable; peak shifts easy to spot | Spectra can overlap and obscure each other |
| **Waterfall (stacked offset)** | Many similar spectra; overlapping lines unreadable | Each trace individually visible | Absolute intensities not comparable across traces |

**Overlay is the standard for food FTIR authentication** in the published literature.
Both are included here; use whichever communicates your data best.

The shaded blue region (1800–650 cm⁻¹) is the **carbohydrate fingerprint** used for PCA.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 · Plot spectra
# Figure 1: Overlay + Waterfall (full spectral range, side by side)
# ─────────────────────────────────────────────────────────────────────────────

n_samples = len(corrected)

# Assign one colour per sample from the 'tab10' qualitative palette.
# np.linspace(0, 0.9, n) spreads n colours evenly across the palette,
# stopping at 0.9 to avoid the grey at the far end (hard to see on white).
# Switch to cm.tab20 if you have more than 10 samples.
colors = cm.tab10(np.linspace(0, 0.9, n_samples))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# ── Left panel: Overlay ───────────────────────────────────────────────────────
ax = axes[0]
for (label, (wn, corr)), col in zip(corrected.items(), colors):
    ax.plot(wn, corr, color=col, linewidth=1.2, label=label)

# FTIR convention: x-axis runs HIGH → LOW wavenumber (high energy on left).
# This matches all published reference spectra so readers can compare directly.
ax.invert_xaxis()
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Absorbance (a.u.)')
ax.set_title('ATR-FTIR Spectra — Overlay (full range)')
ax.legend()

# Lightly shade the fingerprint region to orientate the reader.
# alpha=0.06 is almost transparent so it does not obscure the spectra.
ax.axvspan(1800, 650, alpha=0.06, color='steelblue')
ax.text(1225, ax.get_ylim()[1] * 0.93, 'fingerprint\nregion',
        fontsize=8, ha='center', color='steelblue', alpha=0.9)
ax.set_xlim(4200, 650)

# ── Right panel: Waterfall (stacked / offset) ─────────────────────────────────
ax2 = axes[1]

# offset_step: vertical spacing between successive traces.
# Increase if intense spectra overlap even when stacked;
# decrease for flatter spectra where labels go off the top.
offset_step = 0.15

for i, ((label, (wn, corr)), col) in enumerate(zip(corrected.items(), colors)):
    # Each trace is shifted up by i × offset_step
    ax2.plot(wn, corr + i * offset_step, color=col, linewidth=1.2)
    # Label sits just to the right of the rightmost data point.
    # corr[-1] is the absorbance at the lowest wavenumber (≈650 cm⁻¹),
    # which appears on the right of the inverted x-axis.
    ax2.text(645, corr[-1] + i * offset_step, label,
             fontsize=8, va='center', ha='left', color=col)

ax2.invert_xaxis()
ax2.set_xlabel('Wavenumber (cm⁻¹)')
ax2.set_ylabel('Absorbance + offset (a.u.)')
ax2.set_title('ATR-FTIR Spectra — Waterfall')
ax2.set_xlim(4200, 600)   # extra right margin to fit text labels
ax2.axvspan(1800, 650, alpha=0.06, color='steelblue')
ax2.set_yticks([])         # offset values are meaningless; hide y ticks

plt.tight_layout()
out_path = OUTPUT_DIR / 'honey_ftir_spectra.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 2: Fingerprint region only (1800–650 cm⁻¹)
#
# Zooming in reveals the fine structure that differentiates honey varieties
# and detects adulteration.  Band assignments from the honey FTIR literature:
#   Tewari et al. (2005) J. Agric. Food Chem. 53(16):6488-6494
#   Paradkar & Irudayaraj (2001) Food Chem. 76(2):231-239
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(11, 5))

for (label, (wn, corr)), col in zip(corrected.items(), colors):
    # Boolean mask selects only wavenumber points inside the fingerprint region.
    # This is robust even if the axis does not start or end exactly on 650/1800.
    mask = (wn >= 650) & (wn <= 1800)
    ax.plot(wn[mask], corr[mask], color=col, linewidth=1.4, label=label)

ax.invert_xaxis()   # FTIR convention: high → low
ax.set_xlabel('Wavenumber (cm⁻¹)')
ax.set_ylabel('Absorbance (a.u.)')
ax.set_title('ATR-FTIR — Carbohydrate Fingerprint Region (1800–650 cm⁻¹)')
ax.legend()

# ── Annotate diagnostic honey bands ──────────────────────────────────────────
# Dict maps wavenumber (cm⁻¹) to chemical assignment text.
# The dotted vertical lines help students trace a visible peak to its origin.
bands = {
    1640: 'H₂O bend',
    1415: 'C-H / O-H',
    1155: 'C-O-C (sugars)',
    1080: 'C-OH (glucose)',
    1025: 'C-OH (fructose)',
     930: 'ring vibrations',
}
for wnum, blabel in bands.items():
    ax.axvline(wnum, color='grey', linestyle=':', linewidth=0.8, alpha=0.7)
    # Rotated label just below the top; ha='right' keeps text left of the line
    ax.text(wnum - 3, ax.get_ylim()[1] * 0.97, blabel,
            rotation=90, fontsize=7, va='top', ha='right', color='grey')

plt.tight_layout()
out_path = OUTPUT_DIR / 'honey_ftir_fingerprint.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

## 5 · Principal Component Analysis (PCA)

### Why PCA for FTIR?

Each spectrum has ~1150 wavenumber points in the fingerprint region — far too many to visualise or interpret directly. PCA finds linear combinations of wavenumber channels (principal components, PCs) that capture the most variance across all spectra.
It compresses the high-dimensional data into 2–3 dimensions while retaining the key chemical differences.

| Output | Interpretation |
|--------|---------------|
| **Scores plot** (PC1 vs PC2) | Each spectrum = one point. Clusters = similar chemistry. Separation = real chemical differences. |
| **Loadings plot** | Which wavenumbers drive each PC. Peaks = channels with discriminating power. |
| **Biplot** | Scores and loading vectors in one figure. Arrows point towards samples with high absorbance at that wavenumber. |

### Pre-processing: mean-centring only

We **mean-centre** (subtract the column mean) but do **not** autoscale (divide by the column standard deviation) because flat spectral regions have near-zero standard deviations, and dividing by them would amplify noise to the same level as real peaks.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 · PCA — build data matrix, mean-centre, decompose
# ─────────────────────────────────────────────────────────────────────────────

# ── Spectral region for PCA ───────────────────────────────────────────────────
# Using 1800–650 cm⁻¹ (fingerprint):
#   ✓ Contains the most diagnostic sugar bands
#   ✗ Excludes the broad water O-H stretch (3600–2800 cm⁻¹) which would
#     dominate variance and mask the more informative sugar signals
FP_LOW, FP_HIGH = 650, 1800   # cm⁻¹

# All corrected spectra share the same wavenumber axis; take the first one
labels_list = list(corrected.keys())
wn_first    = corrected[labels_list[0]][0]

# Boolean mask: True for wavenumbers inside the fingerprint region
fp_mask = (wn_first >= FP_LOW) & (wn_first <= FP_HIGH)
wn_fp   = wn_first[fp_mask]   # wavenumber axis for fingerprint region

# ── Assemble the data matrix X ────────────────────────────────────────────────
# Shape: (n_samples, n_wavenumber_points)
# np.array() converts a list of 1-D arrays into a 2-D array by stacking rows.
X = np.array([corr[fp_mask] for _, (wn, corr) in corrected.items()])

print(f"Data matrix X : {X.shape[0]} samples × {X.shape[1]} wavenumber points")

# ── Mean-centring ─────────────────────────────────────────────────────────────
# Subtract the column-wise mean spectrum.
# X.mean(axis=0) → shape (n_features,) which is broadcast over all rows.
X_mc = X - X.mean(axis=0)

# ── PCA ──────────────────────────────────────────────────────────────────────
# PCA cannot produce more non-trivial components than there are samples,
# so we cap n_components at the number of samples.
n_components = min(len(labels_list), 5)
pca    = PCA(n_components=n_components)
# fit_transform:
#   fit      – computes the PC directions (eigenvectors) from X_mc
#   transform – projects each sample onto those directions → score values
scores = pca.fit_transform(X_mc)   # shape: (n_samples, n_components)

explained = pca.explained_variance_ratio_ * 100   # as percentage

print("\nVariance explained per PC:")
for i, ev in enumerate(explained):
    print(f"  PC{i+1}: {ev:5.1f}%  {'█' * int(ev/2)}")
print(f"\n  Cumulative (PC1–PC{n_components}): {explained.sum():.1f}%")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 3: PCA Scores plot  (PC1 vs PC2)
#
# Each point = one honey sample projected onto the first two principal components.
# Samples that are chemically similar cluster together.
# The axes are dimensionless 'score' units; their meaning comes from the loadings.
# ─────────────────────────────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(8, 6))

for i, (label, col) in enumerate(zip(labels_list, colors)):
    # zorder=3 draws markers above grid lines
    # edgecolors='k' adds a thin black outline so light colours remain visible
    ax.scatter(scores[i, 0], scores[i, 1],
               color=col, s=120, zorder=3, edgecolors='k', linewidths=0.5)
    # Offset the label slightly (8 pt right, 4 pt up) to avoid overlapping the marker
    ax.annotate(label, (scores[i, 0], scores[i, 1]),
                textcoords='offset points', xytext=(8, 4),
                fontsize=9, color=col)

# Cross-hairs at the origin (= mean spectrum after mean-centring)
ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.6, linestyle='--')

# Axis labels include % variance explained so readers see the relative importance
ax.set_xlabel(f'PC1  ({explained[0]:.1f} % variance explained)')
ax.set_ylabel(f'PC2  ({explained[1]:.1f} % variance explained)')
ax.set_title('PCA Scores — ATR-FTIR Fingerprint Region (1800–650 cm⁻¹)')

plt.tight_layout()
out_path = OUTPUT_DIR / 'honey_pca_scores.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 4: PCA Loadings plot  (PC1 and PC2 plotted as spectra)
#
# A loading has one value per wavenumber channel.  Plotted against wavenumber
# it looks like a spectrum and reveals which parts of the infrared range drive
# the separation seen in the scores plot:
#   Positive loading → high absorbance here pushes a sample towards the +PC end
#   Negative loading → high absorbance here pushes a sample towards the −PC end
#
# Overlaying the same band markers as Figure 2 enables direct chemical assignment.
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(11, 7),
                          sharex=True)   # linked x-axis: zoom one panel zooms both

for pc_idx, ax in enumerate(axes):
    loading = pca.components_[pc_idx]   # shape (n_wavenumber_points,)

    # Filled regions make the sign of each feature visually obvious at a glance.
    # Using the same colour for both lobes (different alpha) keeps the palette tidy.
    ax.fill_between(wn_fp, loading, 0,
                    where=(loading >= 0), color=f'C{pc_idx}', alpha=0.4)
    ax.fill_between(wn_fp, loading, 0,
                    where=(loading < 0),  color=f'C{pc_idx}', alpha=0.15)
    ax.plot(wn_fp, loading, color=f'C{pc_idx}', linewidth=1.0)

    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.set_ylabel(f'PC{pc_idx + 1} loading\n({explained[pc_idx]:.1f} %)')
    ax.invert_xaxis()

    # Overlay the diagnostic band markers from Figure 2 for cross-reference
    for wnum in bands.keys():
        if FP_LOW <= wnum <= FP_HIGH:
            ax.axvline(wnum, color='grey', linestyle=':', linewidth=0.8, alpha=0.6)

axes[-1].set_xlabel('Wavenumber (cm⁻¹)')
fig.suptitle('PCA Loadings — ATR-FTIR Fingerprint Region (1800–650 cm⁻¹)', fontsize=13)

plt.tight_layout()
out_path = OUTPUT_DIR / 'honey_pca_loadings.png'
plt.savefig(out_path, bbox_inches='tight')
plt.show()
print(f"Saved → {out_path}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Figure 5: PCA Biplot  (scores + top loading vectors)
#
# The biplot overlays the sample scores (dots) with arrows showing which
# wavenumber channels drive the separation.  An arrow pointing towards a
# cluster of samples means those samples have high absorbance at that wavenumber.
#
# Only the 5 channels with the largest absolute PC1 loading are drawn to avoid
# clutter.  This figure is most meaningful with ≥ 4 samples.
# ─────────────────────────────────────────────────────────────────────────────

if len(labels_list) < 4:
    print("Biplot skipped — need ≥ 4 samples for a meaningful biplot.")
    print("Add more sample CSV files to DATA_DIR and re-run the notebook.")
else:
    fig, ax = plt.subplots(figsize=(9, 7))

    # ── Sample scores ─────────────────────────────────────────────────────────
    for i, (label, col) in enumerate(zip(labels_list, colors)):
        ax.scatter(scores[i, 0], scores[i, 1],
                   color=col, s=110, zorder=3, edgecolors='k', linewidths=0.5)
        ax.annotate(label, (scores[i, 0], scores[i, 1]),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)

    # ── Scale loading vectors to fit the scores axes ───────────────────────────
    # Scores and loadings have different units.  We rescale so the longest arrow
    # reaches ~80 % of the maximum score distance from the origin.
    # This is purely cosmetic and does not change which channels are important.
    score_range   = np.max(np.abs(scores[:, :2]))
    loading_range = np.max(np.abs(pca.components_[:2]))
    scale = (0.8 * score_range / loading_range) if loading_range > 0 else 1.0

    # ── Top-5 most influential wavenumber vectors (by |PC1 loading|) ──────────
    # np.argsort returns indices in ascending order; [-5:] takes the last five.
    top5_idx = np.argsort(np.abs(pca.components_[0]))[-5:]

    for idx in top5_idx:
        lx = pca.components_[0, idx] * scale   # scaled x-component
        ly = pca.components_[1, idx] * scale   # scaled y-component
        # Arrow from origin to loading vector tip
        ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='steelblue', lw=1.8))
        # Label just beyond the arrowhead (factor 1.08 nudges it past the tip)
        ax.text(lx * 1.08, ly * 1.08, f'{wn_fp[idx]:.0f} cm⁻¹',
                fontsize=8, color='steelblue', ha='center')

    ax.axhline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.axvline(0, color='grey', linewidth=0.6, linestyle='--')
    ax.set_xlabel(f'PC1  ({explained[0]:.1f} %)')
    ax.set_ylabel(f'PC2  ({explained[1]:.1f} %)')
    ax.set_title('PCA Biplot — sample scores + top-5 PC1 loading wavenumbers\n'
                 '(blue arrows: high absorbance at that wavenumber → arrow direction)')

    plt.tight_layout()
    out_path = OUTPUT_DIR / 'honey_pca_biplot.png'
    plt.savefig(out_path, bbox_inches='tight')
    plt.show()
    print(f"Saved → {out_path}")

## 6 · Summary

| Step | Choice | Rationale |
|------|--------|-----------|
| Unit conversion | %T → Absorbance: A = 2 − log₁₀(%T) | Required for Beer-Lambert linearity; subtraction meaningful only in absorbance |
| Blank correction | Nearest blank by sample number | Blanks recorded interleaved; closest in time gives best background estimate |
| Spectral region for PCA | 1800–650 cm⁻¹ (fingerprint) | Carbohydrate-rich; most diagnostic for honey origin and adulteration |
| PCA pre-processing | Mean-centring only (no variance scaling) | Autoscaling amplifies noise in flat spectral regions |
| Chemometrics | PCA — scores, loadings, biplot | Standard exploratory method for food authentication |

---

### Adding more samples (practical class data)

1. Place additional `.csv` files (same Perkin-Elmer Spectrum export format) in `data/honey/`.
2. Files starting with `blank` are automatically treated as backgrounds.
3. All other filenames become sample labels in the plots — name files descriptively!
4. Re-run all cells (`Runtime → Run all` on Colab, `Kernel → Restart & Run All` locally).

### Output files — saved to `data/processed/`

| Filename | Contents |
|----------|----------|
| `honey_ftir_spectra.png` | Overlay + waterfall, full spectral range |
| `honey_ftir_fingerprint.png` | Fingerprint region with diagnostic band annotations |
| `honey_pca_scores.png` | PCA scores plot (PC1 vs PC2) |
| `honey_pca_loadings.png` | PC1 and PC2 loadings plotted as spectra |
| `honey_pca_biplot.png` | Biplot (scores + top-5 PC1 loading wavenumbers) |